# MetroPT-3: Industrial Failure Prediction, Four-Model Comparison

Predicting compressor failures from sensor data (MetroPT-3 dataset, 1,516,948 rows) with a chronological train/test split and four classifiers: Logistic Regression, Random Forest, XGBoost and LightGBM.

*Part of a manuscript in preparation: "A Machine Learning Framework for Intelligent Anomaly Detection and Predictive Monitoring of Industrial Systems".*

## What this notebook does
- Loads the **final engineered dataset** (43 predictor columns plus a binary target) and checks the column count.
- Splits the data **chronologically** (no shuffling): 856,832 training rows and 660,116 test rows.
- Fits the imputer on the training set only, then trains the four models with the settings documented in the manuscript.
- Evaluates each model at its decision threshold and compares the results with the values reported in the manuscript.

## What it does not include
- The raw-data feature-engineering code is not part of this notebook. It expects the final 43-feature CSV, which is not included in this repository.
- Settings the manuscript does not specify (for example the Logistic Regression settings) are reasonable defaults. A fresh run can differ slightly from the reported numbers.

## 1. Install and import packages

In [ ]:
!pip -q install xgboost lightgbm scikit-learn joblib seaborn

import os
import warnings
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from google.colab import files
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
OUTPUT_DIR = "/content/MetroPT3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Environment ready.")

## 2. Upload the final engineered dataset

Upload the CSV with **43 predictor columns and the binary target column** (the file name should contain `final_43_features`). If the file has no target column, the notebook stops. Do not guess or recreate labels.

In [ ]:
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("No CSV file uploaded. Upload the final engineered dataset.")

preferred = [name for name in csv_files if "final_43_features" in name.lower()]
DATA_FILE = preferred[0] if preferred else csv_files[0]

df = pd.read_csv(DATA_FILE)
print("Loaded file:", DATA_FILE)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Identify the target and timestamp columns

In [ ]:
TARGET_CANDIDATES = ["failure", "target", "label", "class", "anomaly", "is_failure"]
TIME_CANDIDATES = ["timestamp", "time", "datetime", "date"]

TARGET_COL = next((c for c in TARGET_CANDIDATES if c in df.columns), None)
TIMESTAMP_COL = next((c for c in TIME_CANDIDATES if c in df.columns), None)
print("Detected target:", TARGET_COL)
print("Detected timestamp:", TIMESTAMP_COL)

if TARGET_COL is None:
    raise ValueError("Could not identify the target column. Set TARGET_COL manually in this cell.")

LABEL_MAP = {
    "normal": 0, "no failure": 0, "no_failure": 0, "healthy": 0, "0": 0,
    "failure": 1, "failed": 1, "anomaly": 1, "fault": 1, "1": 1
}

def to_binary(series):
    """Convert the target column to integer 0/1 and stop if it is not binary."""
    if series.dtype == "object":
        out = series.astype(str).str.strip().str.lower().map(LABEL_MAP)
    else:
        out = pd.to_numeric(series, errors="coerce")
    if out.isna().any() or not set(out.unique()).issubset({0, 1}):
        raise ValueError("Target must contain only 0 and 1 (or recognised Normal/Failure labels).")
    return out.astype(int)

y_check = to_binary(df[TARGET_COL])
print("Target counts (0 = normal, 1 = failure):")
display(y_check.value_counts().sort_index().rename_axis("target").to_frame("rows"))

## 4. Sort chronologically and build the 43-feature matrix

The timestamp is used only to order the rows. It is not given to the models. Index columns, the target and the timestamp are removed from the predictors. The code **stops if the feature count is not exactly 43**, which protects against training on an index column or leaking the target.

In [ ]:
data = df.copy()

index_cols = [c for c in data.columns
              if c.lower().startswith("unnamed:") or c.lower() in {"index", "row_id"}]
data = data.drop(columns=index_cols, errors="ignore")

if TIMESTAMP_COL is not None:
    data[TIMESTAMP_COL] = pd.to_datetime(data[TIMESTAMP_COL], errors="coerce")
    if data[TIMESTAMP_COL].isna().any():
        raise ValueError("Some timestamps could not be parsed. Clean the timestamp column first.")
    data = data.sort_values(TIMESTAMP_COL, kind="stable").reset_index(drop=True)
else:
    print("WARNING: No timestamp column found. The CSV row order is kept.")
    data = data.reset_index(drop=True)

# Rebuild y after sorting so labels stay aligned with their rows.
y = to_binary(data[TARGET_COL]).reset_index(drop=True)

excluded = [TARGET_COL] + ([TIMESTAMP_COL] if TIMESTAMP_COL is not None else [])
X = data.drop(columns=excluded, errors="ignore")
X = X.select_dtypes(include=[np.number]).copy()
X = X.replace([np.inf, -np.inf], np.nan).reset_index(drop=True)

print("Rows:", len(X), "| Predictor columns:", X.shape[1])

if len(X) != len(y):
    raise ValueError("Feature and target row counts do not match.")
if X.shape[1] != 43:
    raise ValueError(f"Expected 43 predictor columns but found {X.shape[1]}. Check the input file.")
print("CHECK PASSED: exactly 43 predictor variables.")

## 5. Chronological train/test split

- Training: **856,832 rows**
- Testing: **660,116 rows**
- Total: **1,516,948 rows**

The exact boundary is used when the file has the expected row count. For a different row count, the same proportion is used. The time series is never shuffled.

In [ ]:
EXPECTED_TOTAL = 1_516_948
EXPECTED_TRAIN = 856_832

if len(X) == EXPECTED_TOTAL:
    split_idx = EXPECTED_TRAIN
    print("Using the exact documented train/test boundary.")
else:
    split_idx = int(len(X) * EXPECTED_TRAIN / EXPECTED_TOTAL)
    print("WARNING: Row count differs from the documented dataset. Using the same split proportion.")

X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

print("Training shape:", X_train.shape, "| Testing shape:", X_test.shape)
print("Training labels:", y_train.value_counts().sort_index().to_dict())
print("Testing labels:", y_test.value_counts().sort_index().to_dict())

if len(X) == EXPECTED_TOTAL:
    assert X_train.shape == (856832, 43)
    assert X_test.shape == (660116, 43)
    print("CHECK PASSED: documented row counts and 43 features match.")

## 6. Handle missing values

The median imputer is fitted on the **training set only** and then applied to the test set, so no test-set information is used.

In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Missing values after imputation - train:", int(X_train_imp.isna().sum().sum()))
print("Missing values after imputation - test:", int(X_test_imp.isna().sum().sum()))

## 7. Define and train the four classifiers

Settings follow the manuscript:
- **Random Forest:** 100 trees, balanced class weights.
- **XGBoost and LightGBM:** 300 trees, maximum depth 6, learning rate 0.05, subsample 0.8, column sample 0.8.
- **LightGBM** class-imbalance weight: `scale_pos_weight = 76.77362258328039`.
- **Logistic Regression:** the manuscript does not list its settings, so standard scaling and balanced class weights are used here.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="binary:logistic", eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM": LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="binary", scale_pos_weight=76.77362258328039,
        random_state=RANDOM_STATE, verbosity=-1)
}

for model_name, model in models.items():
    print(f"Training {model_name} ...")
    model.fit(X_train_imp, y_train)
print("All four models trained.")

## 8. Evaluate at the manuscript thresholds

Decision thresholds from the manuscript: Logistic Regression **0.50**, Random Forest **0.03**, XGBoost **0.02**, LightGBM **0.01**.

Because failures are rare, **PR-AUC** is the most informative metric here. Accuracy is high for any model that predicts "normal" most of the time.

In [ ]:
THRESHOLDS = {"Logistic Regression": 0.50, "Random Forest": 0.03, "XGBoost": 0.02, "LightGBM": 0.01}

fresh_results, probabilities, predictions = [], {}, {}

for model_name, model in models.items():
    prob = model.predict_proba(X_test_imp)[:, 1]
    pred = (prob >= THRESHOLDS[model_name]).astype(int)
    probabilities[model_name], predictions[model_name] = prob, pred

    fresh_results.append({
        "Model": model_name,
        "Threshold": THRESHOLDS[model_name],
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "PR-AUC": average_precision_score(y_test, prob),
        "Predicted Failures": int(pred.sum())
    })

fresh_results_df = pd.DataFrame(fresh_results)
display(fresh_results_df.round(4))
fresh_results_df.to_csv(os.path.join(OUTPUT_DIR, "model_results.csv"), index=False)

## 9. Compare with the results reported in the manuscript

The table below contains the values **reported in the manuscript (Table 1)**. They are typed in from the original experiment and are not computed by this notebook. The second table shows the difference between this run and the reported values.

In [ ]:
REPORTED_RESULTS = pd.DataFrame([
    ["Logistic Regression", 0.5000, 0.9844, 0.7353, 0.7103, 0.7226, 0.9605, 0.5304, 18294],
    ["Random Forest",       0.0300, 0.9708, 0.4911, 0.5430, 0.5158, 0.8861, 0.4620, 20935],
    ["XGBoost",             0.0200, 0.9748, 0.6546, 0.2548, 0.3668, 0.9687, 0.4717, 7371],
    ["LightGBM",            0.0100, 0.9803, 0.7845, 0.4330, 0.5580, 0.9714, 0.6042, 10452]
], columns=["Model", "Threshold", "Accuracy", "Precision", "Recall",
            "F1-score", "ROC-AUC", "PR-AUC", "Predicted Failures"])

print("Reported in the manuscript:")
display(REPORTED_RESULTS)

metric_cols = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]
diff = fresh_results_df.set_index("Model")[metric_cols] - REPORTED_RESULTS.set_index("Model")[metric_cols]
print("This run minus reported (positive = this run is higher):")
display(diff.round(4))

## 10. Confusion matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (model_name, pred) in zip(axes.ravel(), predictions.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(f"{model_name} (threshold={THRESHOLDS[model_name]})")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Actual class")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices.png"), dpi=300)
plt.show()

## 11. Metric comparison charts

In [ ]:
for metric in ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]:
    plt.figure(figsize=(8, 5))
    sns.barplot(data=fresh_results_df, x="Model", y=metric)
    plt.xticks(rotation=20, ha="right")
    plt.title(metric)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{metric.lower().replace('-', '_')}.png"), dpi=300, bbox_inches="tight")
    plt.show()

## 12. LightGBM threshold sensitivity (descriptive only)

This shows how LightGBM's precision, recall and F1-score change with the threshold **on the test set**. It is a sensitivity view only. Do not use it to choose the threshold, because choosing a threshold on the test set makes the reported scores optimistic. Section 13 shows a validation-based alternative.

In [ ]:
lgbm_prob = probabilities["LightGBM"]
rows = []
for threshold in np.arange(0.001, 0.101, 0.001):
    pred = (lgbm_prob >= threshold).astype(int)
    rows.append({
        "Threshold": round(float(threshold), 3),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "Predicted Failures": int(pred.sum())
    })
threshold_df = pd.DataFrame(rows)
display(threshold_df.sort_values("F1-score", ascending=False).head(10).round(4))
threshold_df.to_csv(os.path.join(OUTPUT_DIR, "lightgbm_threshold_sensitivity.csv"), index=False)

plt.figure(figsize=(9, 5))
for col in ["Precision", "Recall", "F1-score"]:
    plt.plot(threshold_df["Threshold"], threshold_df[col], label=col)
plt.axvline(0.01, linestyle="--", color="grey", label="Manuscript threshold: 0.01")
plt.xlabel("Decision threshold"); plt.ylabel("Metric score")
plt.title("LightGBM threshold sensitivity (test set, descriptive)")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "lightgbm_threshold_sensitivity.png"), dpi=300)
plt.show()

## 13. Robustness check: choose thresholds on a validation slice

To avoid selecting thresholds on the test set, the last 20% of the **training period** is held out as a validation slice. Each model is refitted on the first 80%, its F1-maximising threshold is chosen on the validation slice, and that threshold is then applied once to the test set.

This is an extra check and does not replace the manuscript results. If the validation slice contains no failures, the F1 score is zero for every threshold, so the result is not meaningful.

In [ ]:
val_start = int(len(X_train_imp) * 0.8)
X_fit, X_val = X_train_imp.iloc[:val_start], X_train_imp.iloc[val_start:]
y_fit, y_val = y_train.iloc[:val_start], y_train.iloc[val_start:]
print("Failures in validation slice:", int(y_val.sum()))

grid = np.unique(np.concatenate([np.arange(0.001, 0.05, 0.001), np.arange(0.05, 0.51, 0.01)]))
check_rows = []

for model_name, model in models.items():
    m = clone(model).fit(X_fit, y_fit)
    p_val = m.predict_proba(X_val)[:, 1]
    best_t = max(grid, key=lambda t: f1_score(y_val, (p_val >= t).astype(int), zero_division=0))
    p_test = m.predict_proba(X_test_imp)[:, 1]
    pred = (p_test >= best_t).astype(int)
    check_rows.append({
        "Model": model_name,
        "Threshold (chosen on validation)": round(float(best_t), 3),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "PR-AUC": average_precision_score(y_test, p_test)
    })

validation_check_df = pd.DataFrame(check_rows)
display(validation_check_df.round(4))
validation_check_df.to_csv(os.path.join(OUTPUT_DIR, "validation_threshold_check.csv"), index=False)

## 14. Save the trained models and download the outputs

In [ ]:
model_bundle = {
    "imputer": imputer,
    "models": models,
    "feature_names": list(X_train.columns),
    "thresholds": THRESHOLDS,
    "random_state": RANDOM_STATE
}
joblib.dump(model_bundle, os.path.join(OUTPUT_DIR, "model_bundle.pkl"))

zip_path = shutil.make_archive("/content/MetroPT3_outputs", "zip", OUTPUT_DIR)
files.download(zip_path)

## Notes and limitations

- The dataset is the public MetroPT-3 data after the project's own feature engineering. That engineered file and the feature-engineering code are not included here.
- The thresholds 0.50, 0.03, 0.02 and 0.01 come from the original experiment, where they were chosen to maximise F1. Section 13 repeats the selection on a validation slice to check how much that choice matters.
- No hyperparameter search, confidence intervals or lead-time analysis are included.
- Results come from one chronological split, so they describe this period of the data only.